# Laboratory 5 - Graph Manipulation/Surgery

Open Neural Network Exchange - ONNX is an open format that allows to represent neural networks. It is possible to export ONNX models from most of the frameworks and on the execution/inference side it is also supported by a lot of compilers and specialized optimization frameworks.

We will go through four things today:

0. Investigate the ONNX graph with netron,
1. Learn how to delete and add nodes to existing model,
2. Learn how to split graph and run separate runtime measurements for both splits,
3. Add custom operation to graph and create the code in C++/CUDA that will allow an efficient inference.

A short word on why this course matters. Sometimes the networks that we are trying to optimize for embedded are big and complicated. Usually the optimization tools provided by HW manufacturers have limited number of supported operators and it is not possible to run the desired network just-like-that. It is very helpful to be able to split the bigger network into several parts which makes the problem easier to handle, also replacing or implementing the unsuported operators on our own gives a lot of flexibility during network deployment on embedded hardware. 

**What you should already know:** basic Python, neural network graph representation.

**What you will be able to do after this notebook:** ONNX format, Netron visualization tool, performing graph surgery, creating custom operator for neural network inference.


## 0. Setup

For this laboratory we will need onnx and onnxruntime packages

```
pip3 install onnx
pip3 install onnxruntime==1.30.0
```

We will also use the yolo onnx model `yolov8n.onnx` please download it from https://github.com/yoobright/yolo-onnx/blob/master/yolov8n.onnx

## 1. Investigate model with netron

Please visit the https://netron.app/ webpage click "Open model" and select the downloaded `yolov8n.onnx` file

Familiarize yourself with the features of netron:
- nodes naming convention names/edges shapes,
- nodes properties, attributes, inputs, outputs,
- nodes weights values

Hint: you can use Ctrl+F in netron to look for a particular node by name

### 1.1 Graph surgery - removing nodes

Please familiarize yourself with `onnx` python package, use `load()`, `remove()` and `save()` functions to remove `Conv`, `Sigmoid` and `Mul` nodes of `/model.1`.
Make sure to adjust the input to `/model.2` as well. 

Save the modified graph to new .onnx file and verify in netron

In [ ]:
import onnx

#load model
model = onnx.load("yolov8n.onnx")
graph = model.graph
model.ir_version = 9

#remove from graph
#TODO

#verify consistency and save new graph
onnx.checker.check_model(model)
onnx.save(model, "yolov8n_deleted_nodes.onnx")

### 1.2 Graph surgery - adding intermediate outputs

Sometimes it is really helpfull to be able to extract the output values from some intermediate neural network layers.

Please use the `onnx` functions `make_tensor_value_info` and `output.append()` to create intermediate output from `/model.22/Mul_2` node

Save the modified graph to new .onnx file and verify in netron

In [ ]:
import onnx

#load model
model = onnx.load("yolov8n.onnx")
graph = model.graph
model.ir_version = 9

# Add an additional output
#TODO

# Append the additional output to the model
#TODO

onnx.checker.check_model(model)
onnx.save(model, "yolov8n_intermediate_output.onnx")

### 1.3 Graph surgery - model split

Please familiarize yourself with `onnx.utils.extract_model()` function.

We will use it to split the original `yolov8n` graph into two separate models aroud the `/model.9/cv2/act/Mul` node

Please identify the names of the outputs and skip connections.

Save the 2 splitted graphs to separate new .onnx files and verify in netron

In [ ]:
import onnx

model = onnx.load("yolov8n.onnx")
graph = model.graph
model.ir_version = 9
onnx.checker.check_model(model)
onnx.save(model, "yolov8n_v9.onnx")

#extract submodel 1
#TODO

#extract submodel 2
#TODO

## 2. ONNX model inference

Please familiarize yourself with onnxruntime python package

### 2.1 Inference example

In [ ]:
import numpy as np
import cv2
import onnxruntime    

def prepare_input(file_name, width, height):
    if image_file is None:
        img_bgr = np.random.randint(256,size=(input_h,input_w,3)).astype(np.uint8)
    else:
        img_bgr = cv2.imread(image_file)

    h, w, _ = img_bgr.shape
    img = cv2.resize(img_bgr, (input_h, input_w))
    img = img.astype('float32') / 255.
    img = img.transpose(2, 0, 1)
    img = img.reshape(*input_shape)

    return img

model_file = "yolov8n.onnx"
image_file = None #"image.jpeg"

session = onnxruntime.InferenceSession(model_file, None)

input_names = [input.name for input in session.get_inputs()]
print('Input Names:', input_names)
input_shape = session.get_inputs()[0].shape
print("The model expects input shape: ", input_shape)

output_names = [out.name for out in session.get_outputs()]
print('Output Names:', output_names)

input_h = input_shape[2]
input_w = input_shape[3]

img = prepare_input(image_file, input_w, input_h)

prediction = session.run(output_names, {input_names[0]: img})[0]

print(prediction.shape)
print(prediction)

### 2.2 Inference time measurement

Please modify the example to enable execution time measurement. 
Remember first execution usually requires more time that the next.

In [ ]:
import numpy as np
import cv2
import onnxruntime    
import time

#TODO

### 2.3 Inference time for graph split

Please modify the example to enable execution time measurement for graphs splitted in section `1.3`

In [ ]:
import numpy as np
import cv2
import onnxruntime    
import time

#TODO

Please use constant input or image to veryfi if both original and splitted graph executions provide same results

## 3 Custom operator

The ONNX is just a definition of the graph (nodes, edges, weights). The inference is performed by specialized software which is able to perform the graph operations. It is usually strongly optimized and HW dependent. Usually not all operations are supported by particular HW. It is therefore needed to implement it on your own. Also sometimes there is a possibility to merge some of the operations and write a faster combined function.

### 3.1 Graphs with standard and custom operation

Please open `standard_mul_add.onnx` and `custom_mul_add.onnx` models in netron and compare them.

### 3.2 Infer the standard graph

Please write code that creates dummy input tensors for `standard_mul_add.onnx` and measure the execution time

In [ ]:
#TODO

### 3.3 Custom C++ operator

Please check the code in the custom_op_cpp/ directory and try to build the operator with the following command:

`g++ -shared -o custom_op_library.so -fPIC custom_op_local_function.cc custom_mul_add.cc -I. -I./core/session`

the result should be a `custom_op_library.so` file

In order to load the custom operator to the onnxruntime, please use the code below

In [ ]:
model_file = "custom_mul_add.onnx"

options = onnxruntime.SessionOptions()
options.register_custom_ops_library("./custom_op_cpp/custom_op_library.so")
session = onnxruntime.InferenceSession(model_file, options, None)

Please add the custom op execution to code created in 3.2 in order to compare execution time and results for both graphs

In [ ]:
import numpy as np
import onnxruntime    
import time

input_0 = np.random.rand(1024,1024,32).astype(np.float32)
input_1 = np.random.rand(1024,1024,32).astype(np.float32)

model_file_1 = "standard_mul_add.onnx"
model_file_2 = "custom_mul_add.onnx"
loop_n = 100

#infer first part
#TODO

#infer second part
#TODO

Please check the error between standard and custom operator (`diff` variable)

### 3.4 Custom C++ operator - kernel behaviour

Please look into the code in `custom_mul_add.cc` and modify it to enable the original implementation from `standard_mul_add.onnx`

Make sure the error (`diff`) is acceptably small

### 3.5 Custom C++ operator - execution time

What is the difference in execution time between standard and custom operator?

Please use the `-O3 -mavx` flags and recompile the custom operator.

Is there any difference ?